<a href="https://colab.research.google.com/github/Shahad-Hossain/nasa-image-enhancement/blob/main/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from PIL import Image
import math
from collections import defaultdict
import random
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
input1 = "noisyImage.png"
try:
    img1 = Image.open(input1).convert('RGB') # Convert to RGB to ensure consistent 3-channel access
except FileNotFoundError:
    print(f"Error: The file '{input1}' was not found.")
    exit()

pixels1 = img1.load()

width1, height1 = img1.size

print(str(width1) + " " + str(height1))

In [ ]:
#Gaussian - additive noise

mean = 125
sigma = 25

def gaussian(z):
  partOne = 1 / (math.sqrt(2 * math.pi) * sigma)

  exponent = -1 * (math.pow(z - mean, 2)) / (2 * math.pow(sigma, 2))

  return partOne * math.exp(exponent)

cdf = []
sum = 0
for i in range(256):
  gauDen = gaussian(i)

  sum += gauDen

  cdf.append(sum)

def gaussian_random():
  rng = random.random()

  for i in range(256):
    if rng <= cdf[i]:
      return i

  return -1


#Salt and Pepper - impulse noise

pepper = 0
pepperProb = 0.01
salt = 255
saltProb = 0.01

def sp_random():
  rng = random.random()

  stayProbability = 1 - pepperProb - saltProb
  if (rng < stayProbability):
    return 1;
  elif (rng < stayProbability + pepperProb):
    return pepper
  else:
    return salt;


#Uniform - additive noise

val1 = 100
val2 = 150

probability = 1 / (val2 - val1)
cdf1 = []
sum = 0
for i in range(val1, val2):
  sum += probability
  cdf1.append(sum)

def uniform_random():
  rng = random.random()
  for i in range(len(cdf1)):
    if rng <= cdf1[i]:
      return i + val1


#Exponential - additive noise

a = 0.05

def exponential(z):
  return a * math.exp(-1 * a * z)


sum = 0
for i in range(0, 256):
  sum += exponential(i)

normalizeConstant = sum

def exponential(z):
  return a * math.exp(-1 * a * z) / normalizeConstant

cdf2 = []
sum = 0
for i in range(256):
  expDen = exponential(i)

  sum += expDen

  cdf2.append(sum)

def exponential_random():
  rng = random.random()

  for i in range(256):
    if rng <= cdf2[i]:
      return i

  return -1


#Rayleigh - additive noise
a1 = 0
b1 = 1

def rayleigh(z):
  z = z / 8
  exponent = -1 * (math.pow(z - a1, 2) / b1)

  return (2 / b1) * (z - a1) * math.exp(exponent)


normalizeConstantRay = 0
for i in range(256):
  normalizeConstantRay += rayleigh(i)

def rayleigh(z):
  z = z / 8
  exponent = -1 * (math.pow(z - a1, 2) / b1)

  return (2 / b1) * (z - a1) * math.exp(exponent) / normalizeConstantRay

cdf3 = []
sum = 0
for i in range(256):
  sum += rayleigh(i)

  cdf3.append(sum)

def rayleigh_random():
  rng = random.random()

  for i in range(256):
    if rng <= cdf3[i]:
      return i

  return -1

In [ ]:
#Gaussian Noise Generator

noiseImg = Image.new(mode="L", size=(width1, height1))

noisePixels = noiseImg.load()
for i in range(width1):
  for j in range(height1):
    noisePixels[i, j] = gaussian_random()


noiseImg.save("noise.jpg")

In [ ]:
#Salt and Pepper Noise Generator

noiseImg = Image.new(mode="L", size=(width1, height1))

noisePixels = noiseImg.load()
for i in range(width1):
  for j in range(height1):
    noisePixels[i, j] = sp_random()



noiseImg.save("noise.jpg")

In [ ]:
#Uniform Noise Generator

noiseImg = Image.new(mode="L", size=(width1, height1))

noisePixels = noiseImg.load()
for i in range(width1):
  for j in range(height1):
    noisePixels[i, j] = uniform_random()


noiseImg.save("noise.jpg")

In [ ]:
#Exponential Noise Generator

noiseImg = Image.new(mode="L", size=(width1, height1))

noisePixels = noiseImg.load()
for i in range(width1):
  for j in range(height1):
    noisePixels[i, j] = exponential_random()


noiseImg.save("noise.jpg")

In [ ]:
#Rayleigh Noise Generator

noiseImg = Image.new(mode="L", size=(width1, height1))

noisePixels = noiseImg.load()
for i in range(width1):
  for j in range(height1):
    noisePixels[i, j] = rayleigh_random()


noiseImg.save("noise.jpg")

In [ ]:
#Logarithmic Addition (assuming greyscaled)

M = 255

outImg = Image.new(mode="L", size=(width1, height1))

outPixels = outImg.load()
for i in range(width1):
  for j in range(height1):
    pixelVal1 = pixels1[i, j][0]
    pixelVal2 = noisePixels[i, j]

    newPixelVal = int(pixelVal1 + pixelVal2 - ((pixelVal1 * pixelVal2) / M))

    outPixels[i, j] = 255 - newPixelVal

outImg.save("output.jpg")

In [ ]:
#Invert image aka g(i, j) for logarithmic image processing

for i in range(width1):
  for j in range(height1):
    pixelVal1 = pixels1[i, j][0]
    pixelVal2 = noisePixels[i, j]

    pixels1[i, j] = (255 - pixelVal1, 255 - pixelVal1, 255 - pixelVal1)
    noisePixels[i, j] = 255 - pixelVal2

In [ ]:
#Add
outImg = Image.new(mode="L", size=(width1, height1))

outPixels = outImg.load()
for i in range(width1):
  for j in range(height1):
    pixelVal1 = pixels1[i, j][0]
    pixelVal2 = noisePixels[i, j]

    newPixelVal = pixelVal1 + pixelVal2

    outPixels[i, j] = newPixelVal

outImg.save("output.jpg")

In [ ]:
#Multiply
outImg = Image.new(mode="L", size=(width1, height1))

outPixels = outImg.load()
for i in range(width1):
  for j in range(height1):
    pixelVal1 = pixels1[i, j][0]
    pixelVal2 = noisePixels[i, j]

    newPixelVal = pixelVal1 * pixelVal2

    outPixels[i, j] = min(255, newPixelVal)

outImg.save("output.jpg")

### **Estimating noise parameters**

In [ ]:
#Zoom into segment of image
startx = 50
endx = 300
starty = 1000
endy = height1 - 250
zoomImg = Image.new(mode="L", size=(endx - startx, endy - starty))

zoomPixels = zoomImg.load()

outPixels = outImg.load()
for i in range(startx, endx):
  for j in range(starty, endy):
    pixelVal1 = pixels1[i, j][0]

    zoomPixels[i - startx, j - starty] = pixelVal1

zoomImg.save("zoom.jpg")

In [ ]:
histogram = defaultdict(int)
size = 0
zoomMean = 0
zoomVariance = 0

for i in range(endx - startx):
  for j in range(endy - starty):
    histogram[zoomPixels[i, j]] += 1

histogramArray = []
for i in range(256):
  histogramArray.append(histogram[i])

for i in range(256):
  zoomMean += i * histogramArray[i]
  size += histogramArray[i]

zoomMean /= size

for i in range(256):
  zoomVariance += math.pow(i - zoomMean, 2) * (histogramArray[i] / size)

print(str(zoomMean) + " " + str(zoomVariance))

In [ ]:
# ---- zoom histogram ----
histogram = defaultdict(int)

for i in range(endx - startx):
  for j in range(endy - starty):
    histogram[zoomPixels[i, j]] += 1

histogramArray = []

for i in range(256):
  histogramArray.append(histogram[i])
# ---- plot histogram as bars ----
plt.figure(figsize=(8,4))
plt.bar(range(256), histogramArray)
plt.title("Image Histogram")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.show()
print(histogramArray[1])

In [ ]:
#Rayleigh noise curve estimation

graphb = (4 * zoomVariance) / (4 - math.pi)
grapha = zoomMean - (math.sqrt((math.pi * graphb) / 4))

def rayleighGraph(z):
  exponent = -1 * (np.power(z - grapha, 2) / graphb)

  return (2 / graphb) * (z - grapha) * np.exp(exponent)

normalizedHistogramArray = []

max = -1
for i in range(256):
  normalizedHistogramArray.append(histogramArray[i] / size)
  if (normalizedHistogramArray[i] > max):
    max = normalizedHistogramArray[i]

# Define the range of x values
x = np.linspace(0, 256, 400)  # 400 points between -10 and 10

# Define the function
y = rayleighGraph(x)

plt.figure(figsize=(8,4))
plt.bar(range(256), normalizedHistogramArray, color='gray', alpha=0.6, label='Histogram')


# Plot the function
plt.plot(x, y, label='y = x^2')

# Add labels and title
plt.xlabel('x')
plt.xlim(0, 255)
plt.ylim(0, max + 0.01)
plt.ylabel('y')
plt.title('Plot of y = x^2')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
#Gaussian noise curve estimation

grapha = zoomMean
graphb = math.sqrt(zoomVariance)

def gaussianGraph(z):
  partOne = 1 / (np.sqrt(2 * math.pi) * graphb)

  exponent = -1 * (np.power(z - grapha, 2)) / (2 * np.power(graphb, 2))

  return partOne * np.exp(exponent)

normalizedHistogramArray = []

max = -1
for i in range(256):
  normalizedHistogramArray.append(histogramArray[i] / size)
  if (normalizedHistogramArray[i] > max):
    max = normalizedHistogramArray[i]

# Define the range of x values
x = np.linspace(0, 256, 400)  # 400 points between -10 and 10

# Define the function
y = gaussianGraph(x)

plt.figure(figsize=(8,4))
plt.bar(range(256), normalizedHistogramArray, color='gray', alpha=0.6, label='Histogram')


# Plot the function
plt.plot(x, y, label='y = x^2')

# Add labels and title
plt.xlabel('x')
plt.xlim(0, 255)
plt.ylim(0, max + 0.01)
plt.ylabel('y')
plt.title('Plot of y = x^2')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
#Exponential noise curve estimation

grapha = 1 / np.sqrt(zoomVariance)
functionNormalizeFactor = 0
def exponentialGraph(z):
  return grapha * np.exp(-1 * grapha * z)

for i in range(256):
  functionNormalizeFactor += exponentialGraph(i)

def exponentialGraph(z):
  return grapha * np.exp(-1 * grapha * z) / functionNormalizeFactor

normalizedHistogramArray = []

max = -1
for i in range(256):
  normalizedHistogramArray.append(histogramArray[i] / size)
  if (normalizedHistogramArray[i] > max):
    max = normalizedHistogramArray[i]

# Define the range of x values
x = np.linspace(0, 256, 400)  # 400 points between -10 and 10

# Define the function
y = exponentialGraph(x)

plt.figure(figsize=(8,4))
plt.bar(range(256), normalizedHistogramArray, color='gray', alpha=0.6, label='Histogram')


# Plot the function
plt.plot(x, y, label='y = x^2')

# Add labels and title
plt.xlabel('x')
plt.xlim(0, 256)
plt.ylim(0, max + 0.01)
plt.ylabel('y')
plt.title('Plot of y = x^2')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

print(functionNormalizeFactor)